# データをParquetに変換してHugging Faceへアップロード

JSONL / JSON（レコードの配列）/ CSV / Parquetを読み込み、必要に応じてシャッフル・train/validation分割してアップロードします。
データセット固有のカラム変換・項目削除・ID変更は行いません。型は入力から自動判定します。
READMEは生成・アップロードしません。

プロジェクトのPython環境をカーネルに選び、設定セルを編集して上から実行してください。
**最後のセルを実行すると、実際にアップロードします。** Parquetの作成だけなら、その前で止めてください。


In [ ]:
# 依存関係がない環境でのみ、コメントを外して実行してください。
# %pip install datasets pyarrow huggingface_hub python-dotenv


## 1. 設定

相対パスは `PROJECT_DIR` を基準にします。入力ファイルとアップロード先を変更してください。
`VALID_RATIO = 0` なら分割せずtrainのみ、`SHUFFLE = False` なら入力順を維持します。
検証用splitの標準名は `validation` です。`valid` にしたい場合は `VALID_SPLIT_NAME` を変更できます。


In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()  # このノートブックを置いたプロジェクトのルート
INPUT_FILES = ["path/to/dataset.jsonl"]  # 同じ形式・カラムのファイルを複数指定できます
OUTPUT_DIR = PROJECT_DIR / "test_output/parquet_upload"
REPO_ID = "your-account/your-dataset"

SHUFFLE = True
SEED = 42
VALID_RATIO = 0.1  # 検証用の割合（0以上、1未満）
VALID_SPLIT_NAME = "validation"  # "valid" も指定可能
PRIVATE = False  # 新規リポジトリの公開設定。既存リポジトリの設定は変更しません


## 2. 読み込み

指定した複数ファイルは指定順で結合します。CSVのセルの型や欠損値、ネストしたJSONの型は自動推定されるので、表示されるスキーマを確認してください。


In [ ]:
from datasets import DatasetDict, load_dataset

paths = [(PROJECT_DIR / Path(path)).resolve() for path in INPUT_FILES]
if not paths:
    raise ValueError("INPUT_FILES を指定してください")
for path in paths:
    if not path.is_file():
        raise FileNotFoundError(path)

formats = {".jsonl": "json", ".json": "json", ".csv": "csv", ".parquet": "parquet"}
input_formats = {formats.get(path.suffix.lower()) for path in paths}
if len(input_formats) != 1 or None in input_formats:
    raise ValueError("JSONL/JSON、CSV、Parquetのうち同じ形式のファイルを指定してください")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
dataset = load_dataset(
    input_formats.pop(),
    data_files=[str(path) for path in paths],
    split="train",
    cache_dir=str(OUTPUT_DIR / ".cache"),
)
if len(dataset) == 0:
    raise ValueError("入力データが空です")
print(f"件数: {len(dataset):,}")
print(dataset.features)


## 3. シャッフル・分割

シャッフルする場合はseedを固定します。分割時は末尾のレコードを検証用に使います。
検証件数は `int(総件数 × VALID_RATIO)`（最低1件、最大総件数−1件）です。
レコード単位の分割なので、同じ質問・記事に属するデータが両splitに入る場合があります。


In [ ]:
if not 0 <= VALID_RATIO < 1:
    raise ValueError("VALID_RATIO は0以上、1未満にしてください")
if VALID_SPLIT_NAME not in {"valid", "validation"}:
    raise ValueError("VALID_SPLIT_NAME は valid または validation にしてください")

ordered = dataset.shuffle(seed=SEED) if SHUFFLE else dataset
if VALID_RATIO == 0:
    splits = DatasetDict({"train": ordered})
else:
    if len(ordered) < 2:
        raise ValueError("分割には2件以上のデータが必要です")
    valid_count = max(1, min(int(len(ordered) * VALID_RATIO), len(ordered) - 1))
    boundary = len(ordered) - valid_count
    splits = DatasetDict({
        "train": ordered.select(range(boundary)),
        VALID_SPLIT_NAME: ordered.select(range(boundary, len(ordered))),
    })

print(splits)


## 4. Parquetファイルを作成

`data/train-00000-of-00001.parquet` などを生成します。同じ出力パスのファイルは上書きします。
最後に再読込して件数を確認します。アップロード対象は今回作成したファイルだけです。


In [ ]:
import pyarrow.parquet as pq

parquet_files = [OUTPUT_DIR / "data" / f"{name}-00000-of-00001.parquet" for name in splits]
if any(path.resolve() in paths for path in parquet_files):
    raise ValueError("出力先が入力ファイルと一致しています。OUTPUT_DIR を変更してください")

(OUTPUT_DIR / "data").mkdir(parents=True, exist_ok=True)
for (name, split), path in zip(splits.items(), parquet_files):
    split.to_parquet(path)
    rows = pq.ParquetFile(path).metadata.num_rows
    if rows != len(split):
        raise ValueError(f"保存後の件数が一致しません: {path}")
    print(f"{name}: {rows:,}件 → {path}")


## 5. Hugging Faceへアップロード

`PROJECT_DIR/.env` の `HF_TOKEN` を読み込みます（設定済みの環境変数を優先）。保存済みのHugging Faceログインも利用できます。
以下のセルはParquetファイルだけを1コミットでアップロードします。同名のファイルは更新し、それ以外の既存ファイルは削除しません。
既存リポジトリの分割構成を変える場合は、古いsplitファイルやREADMEの設定も別途合わせてください。


In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import HfApi

if REPO_ID == "your-account/your-dataset" or len(REPO_ID.split("/")) != 2:
    raise ValueError("REPO_ID を account/dataset-name 形式で設定してください")

load_dotenv(PROJECT_DIR / ".env", override=False)
api = HfApi(token=os.environ.get("HF_TOKEN") or None)
api.create_repo(repo_id=REPO_ID, repo_type="dataset", private=PRIVATE, exist_ok=True)
commit = api.upload_folder(
    repo_id=REPO_ID,
    repo_type="dataset",
    folder_path=OUTPUT_DIR,
    allow_patterns=[path.relative_to(OUTPUT_DIR).as_posix() for path in parquet_files],
    commit_message="Upload Parquet dataset",
)
print(f"アップロード完了: https://huggingface.co/datasets/{REPO_ID}")
print(f"コミット: {commit.commit_url}")
